In [4]:
# THU THẬP DỮ LIỆU TẤT CẢ LAPTOP TẠI PHONG VŨ
import pandas as pd
import time
import re
import requests
from bs4 import BeautifulSoup

# Giả lập trình duyệt để tránh bị chặn khi cào dữ liệu
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def phan_loai_card(vga_str):
    vga_str = str(vga_str).upper()
    tu_khoa_card_roi = ['RTX', 'GTX', 'RX ', 'RADEON PRO', 'ARC']
    if any(tu_khoa in vga_str for tu_khoa in tu_khoa_card_roi):
        return 'Card Rời'
    return 'Card Onboard'

# Hàm bóc tách thông tin chi tiết của 1 sản phẩm Laptop
def extract_laptop_info(card):
    try:
        title_elem = card.select_one("div.att-product-card-title h3")
        if not title_elem: return None
            
        ten_raw = title_elem.text.strip()
        ten = ten_raw.split('(')[0].strip()
        
        a_tag = card.select_one("a")
        url_sp = a_tag["href"] if a_tag and "href" in a_tag.attrs else ""
        if url_sp and not url_sp.startswith("http"):
            url_sp = "https://phongvu.vn" + url_sp
            
        try:
            gia_ht_elem = card.select_one("div.att-product-detail-latest-price")
            gia_hien_tai = int(re.sub(r"[^\d]", "", gia_ht_elem.text)) if gia_ht_elem else 0
        except:
            gia_hien_tai = 0

        try:
            gia_goc_elem = card.select_one("del") or card.select_one(".retail-price") or card.select_one("div.att-product-detail-retail-price")
            gia_goc = int(re.sub(r"[^\d]", "", gia_goc_elem.text)) if gia_goc_elem else gia_hien_tai
        except:
            gia_goc = gia_hien_tai 

        if gia_goc < gia_hien_tai:
            gia_goc = gia_hien_tai
            
        muc_giam = gia_goc - gia_hien_tai
        phan_tram_giam = f"{round((muc_giam / gia_goc * 100), 1):g}%" if gia_goc > 0 and muc_giam > 0 else "0%"

        # Gộp toàn bộ text trong card để quét Regex thông số cấu hình
        card_text = card.get_text(separator=" ", strip=True)

        ram_match = re.search(r'\b(4|8|16|24|32|64)\s*GB\b', card_text, re.IGNORECASE)
        ram = ram_match.group(0).upper() if ram_match else "N/A"

        ssd_match = re.search(r'\b(128|256|512)\s*GB\b|\b(1|2)\s*TB\b', card_text, re.IGNORECASE)
        ssd = ssd_match.group(0).upper() if ssd_match else "N/A"

        # Đã nâng cấp pattern để bắt thêm dòng chip Ultra (VD: Ultra 5-225H)
        cpu_pattern = r'\b(i[3579]-?\w+|U[579]-?\w+|Ultra\s?[579]-?\w+|R[3579]\s?\w+|Core\s?[357]\s?\w+|Apple\s?M\d\w*)\b'
        cpu_match = re.search(cpu_pattern, card_text, re.IGNORECASE)
        cpu = cpu_match.group(0) if cpu_match else "N/A"

        gpu_pattern = r'\b(RTX\s?\d{4}|GTX\s?\d{4}|Radeon\w*|Intel\s?Graphics|Arc\s?Graphics|Apple\s?GPU)\b'
        gpu_match = re.search(gpu_pattern, card_text, re.IGNORECASE)
        gpu = gpu_match.group(0) if gpu_match else "Onboard"
        
        trong_luong_match = re.search(r'\b\d+(?:\.\d+)?\s*(?:kg|g)\b', card_text, re.IGNORECASE)
        trong_luong = trong_luong_match.group(0) if trong_luong_match else "N/A"

        man_hinh_match = re.search(r'\b\d+(?:\.\d+)?\s*["”]\s*[a-zA-Z0-9/\s]+(?:Hz|IPS|OLED|WVA)', card_text, re.IGNORECASE)
        man_hinh = man_hinh_match.group(0).strip() if man_hinh_match else "N/A"

        loai_card = phan_loai_card(gpu)

        if ten and gia_hien_tai > 0:
            return {
                "Tên sản phẩm": ten, 
                "CPU": cpu, 
                "RAM": ram, 
                "SSD": ssd, 
                "VGA/GPU": gpu,
                "Loại Card": loai_card,
                "Màn hình": man_hinh,
                "Trọng lượng": trong_luong,
                "Giá gốc": gia_goc, 
                "Giá hiện tại": gia_hien_tai, 
                "Mức giảm": muc_giam, 
                "% Giảm giá": phan_tram_giam, 
                "URL": url_sp
            }
        return None
    except Exception:
        return None

# Hàm cào dữ liệu qua nhiều trang
def scrape_phong_vu_data(so_luong_can_lay, url_template, mo_ta_chuyen_muc):
    print(f"--- Đang bắt đầu thu thập dữ liệu: {mo_ta_chuyen_muc} ---")
    results = []
    page = 1

    while len(results) < so_luong_can_lay:
        url = url_template.format(page=page)
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            if response.status_code != 200: break
                
            soup = BeautifulSoup(response.content, "html.parser")
            cards = soup.select("div.product-card") or soup.select(".css-137u7ef")
            
            if not cards:
                print(f"Hết sản phẩm ở trang {page}.")
                break

            for card in cards:
                if len(results) >= so_luong_can_lay: break
                info = extract_laptop_info(card)
                if info:
                    results.append(info)
                    print(f"[{len(results)}] {info['Tên sản phẩm']} | {info['Giá hiện tại']}đ (-{info['% Giảm giá']})")
            
            page += 1
            time.sleep(1.5)
            
        except Exception as e:
            print(f"Lỗi: {e}")
            break

    return pd.DataFrame(results)

if __name__ == "__main__":
    SO_LUONG = 30
    URL_LAPTOP = "https://phongvu.vn/c/laptop?page={page}"

    df_laptop = scrape_phong_vu_data(SO_LUONG, URL_LAPTOP, "Tất cả Laptop Phong Vũ")
    
    if not df_laptop.empty:
        df_laptop.to_csv("Laptop_PhongVu.csv", index=False, encoding='utf-8-sig')
        print(f"\nĐã lưu {len(df_laptop)} sản phẩm vào file 'Laptop_PhongVu.csv'")

--- Đang bắt đầu thu thập dữ liệu: Tất cả Laptop Phong Vũ ---
[1] Laptop Lenovo IdeaPad Slim 3 14IRH10 - 83K0004JVN | 20590000đ (-1.9%)
[2] Laptop HP 15-fc0024AU - D0BH2PA | 16490000đ (-6.3%)
[3] Laptop Acer Aspire 7 A715-59G-55MD | 25490000đ (-12.1%)
[4] Laptop Asus Vivobook S14 S3407CA-LY068WS | 23490000đ (-16.1%)
[5] Laptop Msi Katana B14WEK-286VN | 33190000đ (-21%)
[6] Laptop Dell 14 DC14250 DC4C5386W | 21990000đ (-8.3%)
[7] Laptop HP 240R G9 - AX3C6AT | 14490000đ (-9.4%)
[8] Laptop Acer Aspire Go 14 AG14-72P-563L | 19990000đ (-20%)
[9] Laptop Asus TUF Gaming A15 FA506NCG-HN184W | 21990000đ (-5.6%)
[10] Laptop HP Victus 15 fa2731TX | 27190000đ (-9.3%)
[11] Laptop Asus Vivobook 14 X1404VA-EB355W | 22290000đ (-14.2%)
[12] Laptop HP Victus 15-fb3116AX - BX8U4PA | 27690000đ (-6.1%)
[13] Laptop Acer Aspire 5 A515-58P-9841 | 21690000đ (-8.1%)
[14] Laptop HP 250R G10 - C3SH7AT | 20990000đ (-12.5%)
[15] Laptop HP OmniBook 5 AI 16-af1054TU - C1MN8PA | 32690000đ (-3.8%)
[16] Laptop Asus Vivo

In [5]:
# THU THẬP DANH SÁCH LAPTOP BÁN CHẠY NHẤT TẠI PHONG VŨ
import pandas as pd
import time
import re
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def extract_laptop_info(card):
    try:
        title_elem = card.select_one("div.att-product-card-title h3")
        if not title_elem: return None
            
        ten_raw = title_elem.text.strip()
        ten = ten_raw.split('(')[0].strip()
        
        a_tag = card.select_one("a")
        url_sp = a_tag["href"] if a_tag and "href" in a_tag.attrs else ""
        if url_sp and not url_sp.startswith("http"):
            url_sp = "https://phongvu.vn" + url_sp
            
        try:
            gia_ht_elem = card.select_one("div.att-product-detail-latest-price")
            gia_hien_tai = int(re.sub(r"[^\d]", "", gia_ht_elem.text)) if gia_ht_elem else 0
        except:
            gia_hien_tai = 0

        try:
            gia_goc_elem = card.select_one("del") or card.select_one(".retail-price") or card.select_one("div.att-product-detail-retail-price")
            gia_goc = int(re.sub(r"[^\d]", "", gia_goc_elem.text)) if gia_goc_elem else gia_hien_tai
        except:
            gia_goc = gia_hien_tai 

        if gia_goc < gia_hien_tai:
            gia_goc = gia_hien_tai
            
        muc_giam = gia_goc - gia_hien_tai
        phan_tram_giam = f"{round((muc_giam / gia_goc * 100), 1):g}%" if gia_goc > 0 and muc_giam > 0 else "0%"

        card_text = card.get_text(separator=" ", strip=True)

        ram_match = re.search(r'\b(4|8|16|24|32|64)\s*GB\b', card_text, re.IGNORECASE)
        ram = ram_match.group(0).upper() if ram_match else "N/A"

        ssd_match = re.search(r'\b(128|256|512)\s*GB\b|\b(1|2)\s*TB\b', card_text, re.IGNORECASE)
        ssd = ssd_match.group(0).upper() if ssd_match else "N/A"

        cpu_pattern = r'\b(i[3579]-?\w+|U[579]-?\w+|R[3579]\s?\w+|Core\s?[357]\s?\w+|Apple\s?M\d\w*)\b'
        cpu_match = re.search(cpu_pattern, card_text, re.IGNORECASE)
        cpu = cpu_match.group(0) if cpu_match else "N/A"

        gpu_pattern = r'\b(RTX\s?\d{4}|GTX\s?\d{4}|Radeon\w*|Intel\s?Graphics|Arc\s?Graphics|Apple\s?GPU)\b'
        gpu_match = re.search(gpu_pattern, card_text, re.IGNORECASE)
        gpu = gpu_match.group(0) if gpu_match else "Onboard"

        if ten and gia_hien_tai > 0:
            return {
                "Tên sản phẩm": ten, 
                "CPU": cpu, 
                "RAM": ram, 
                "SSD": ssd, 
                "VGA/GPU": gpu,
                "Giá gốc": gia_goc, 
                "Giá hiện tại": gia_hien_tai, 
                "Mức giảm": muc_giam, 
                "% Giảm giá": phan_tram_giam, 
                "URL": url_sp
            }
        return None
    except Exception:
        return None

def scrape_phong_vu_data(so_luong_can_lay, url_template, mo_ta_chuyen_muc):
    print(f"--- Đang bắt đầu thu thập dữ liệu: {mo_ta_chuyen_muc} ---")
    results = []
    page = 1

    while len(results) < so_luong_can_lay:
        url = url_template.format(page=page)
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            if response.status_code != 200: break
                
            soup = BeautifulSoup(response.content, "html.parser")
            cards = soup.select("div.product-card") or soup.select(".css-137u7ef")
            
            if not cards:
                print(f"Hết sản phẩm ở trang {page}.")
                break

            for card in cards:
                if len(results) >= so_luong_can_lay: break
                info = extract_laptop_info(card)
                if info:
                    results.append(info)
                    print(f"[{len(results)}] {info['Tên sản phẩm']} | {info['Giá hiện tại']}đ (-{info['% Giảm giá']})")
            
            page += 1
            time.sleep(1.5)
            
        except Exception as e:
            print(f"Lỗi: {e}")
            break

    return pd.DataFrame(results)

if __name__ == "__main__":
    SO_LUONG = 30

    URL_LAPTOP_BAN_CHAY = "https://phongvu.vn/c/laptop?sort=SORT_BY_TOP_SALE_QUANTITY_7_DAYS&order=DESC&page={page}"

    df_laptop = scrape_phong_vu_data(SO_LUONG, URL_LAPTOP_BAN_CHAY, "Laptop Bán Chạy Nhất Phong Vũ")
    
    if not df_laptop.empty:
        df_laptop.to_csv("Laptop_BanChay_PhongVu.csv", index=False, encoding='utf-8-sig')
        print(f"\nĐã lưu {len(df_laptop)} sản phẩm vào file 'Laptop_BanChay_PhongVu.csv'")

--- Đang bắt đầu thu thập dữ liệu: Laptop Bán Chạy Nhất Phong Vũ ---
[1] Laptop Lenovo IdeaPad Slim 3 14IRH10 - 83K0004JVN | 20990000đ (-0%)
[2] Laptop HP 15-fc0024AU - D0BH2PA | 16490000đ (-6.3%)
[3] Laptop Acer Aspire 7 A715-59G-55MD | 25490000đ (-12.1%)
[4] Laptop Asus Vivobook S14 S3407CA-LY068WS | 23490000đ (-16.1%)
[5] Laptop Msi Katana B14WEK-286VN | 33190000đ (-21%)
[6] Laptop Dell 14 DC14250 DC4C5386W | 21990000đ (-8.3%)
[7] Laptop HP 240R G9 - AX3C6AT | 14490000đ (-9.4%)
[8] Laptop Acer Aspire Go 14 AG14-72P-563L | 19990000đ (-20%)
[9] Laptop Asus TUF Gaming A15 FA506NCG-HN184W | 21990000đ (-5.6%)
[10] Laptop HP Victus 15 fa2731TX | 27190000đ (-9.3%)
[11] Laptop Asus Vivobook 14 X1404VA-EB355W | 22290000đ (-14.2%)
[12] Laptop HP Victus 15-fb3116AX - BX8U4PA | 27690000đ (-6.1%)
[13] Laptop Acer Aspire 5 A515-58P-9841 | 21690000đ (-8.1%)
[14] Laptop HP 250R G10 - C3SH7AT | 20990000đ (-12.5%)
[15] Laptop HP OmniBook 5 AI 16-af1054TU - C1MN8PA | 32690000đ (-3.8%)
[16] Laptop Asus